# Logic-Zero · Task 7: SFT Training

Fine-tune Qwen2.5-1.5B with LoRA on the 1183-example SFT dataset built in Task 6 (`data/sft_data.jsonl`). Saves the best-by-dev-accuracy checkpoint to `results/checkpoints/sft/best/` and backs it up to Drive.

**Required runtime:** GPU — L4 (recommended, bf16) or T4 (works in fp16, ~30% slower).

**Expected duration:** 1.5–2 h on L4, 2.5–3 h on T4 (3 epochs × ~1180 examples × batch 4 × grad-accum 4 + 3 × 170-puzzle dev-eval rounds).

**Cost:** purely Colab compute. No external API calls.

**Before running:**
- Confirm Runtime type → **GPU**
- `WANDB_API_KEY` in Colab Secrets (optional — enables loss curve logging)

**If you previously hit `torchvision::nms does not exist`:** you've found the right notebook — the install cell below avoids fighting Colab's preinstalled torch.

## 1. Verify GPU + precision support

In [ ]:
!nvidia-smi | head -20
import torch
print(f'\ntorch {torch.__version__}  CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'bf16 supported: {torch.cuda.is_bf16_supported()}  (T4=False, L4/A100=True)')
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM: {free/1e9:.1f}GB free / {total/1e9:.1f}GB total')

## 2. Clone repo

Idempotent: if `/content/logic-zero` exists from a prior run in this session, just `cd` in.  Data files (`sft_data.jsonl`, `eval_data.jsonl`, `dev_data.jsonl`, `eval_hashes.json`) are now committed to the repo, so no separate download step.

In [ ]:
import os
if not os.path.isdir('/content/logic-zero'):
    !git clone https://github.com/bsdnn/logic-zero.git /content/logic-zero
%cd /content/logic-zero
!git pull origin main

## 3. Install dependencies (DO NOT downgrade Colab's torch)

**Strategy:** keep Colab's preinstalled `torch` + `torchvision` (currently 2.10+) and install only the missing packages on top.  `requirements.txt` pins `torch==2.4.0` which breaks the preinstalled torchvision — we skip it here.

Robust to flaky pypi: 3 retries × 180 s timeout per package.

In [ ]:
import subprocess

def pip_inst(specs, retries=3):
    for i in range(retries):
        r = subprocess.run(
            ['pip', 'install', '-q', '--default-timeout=180'] + specs,
            capture_output=True, text=True,
        )
        if r.returncode == 0:
            print(f'✓ {specs}')
            return
        print(f'[retry {i+1}/{retries}] {r.stderr[-300:]}')
    raise RuntimeError(f'pip install failed after {retries} retries: {specs}')

# Compatible with Colab's torch 2.10:
pip_inst(['trl>=0.13.0,<0.15.0', 'peft>=0.13.0,<0.16.0',
          'datasets>=3.0.0,<4.0.0', 'accelerate>=1.0.0'])
pip_inst(['z3-solver', 'python-dotenv', 'openai>=1.40', 'wandb'])

# Verify
import torch, transformers, trl, peft, z3, openai
print(f'\ntorch={torch.__version__}  transformers={transformers.__version__}')
print(f'trl={trl.__version__}  peft={peft.__version__}')
print(f'z3={z3.get_version_string()}  openai={openai.__version__}')
# Sanity import the thing that errored last time
from transformers.models.bloom import modeling_bloom  # noqa: F401
print('✓ torchvision::nms not triggered — imports clean')

## 4. Secrets → `.env` (WANDB optional)

WANDB key is optional; without it, training still runs but no loss curves are logged.

In [ ]:
from google.colab import userdata
lines = []
for key in ['WANDB_API_KEY', 'HF_TOKEN']:
    try:
        val = userdata.get(key)
        if val:
            lines.append(f'{key}={val}')
            os.environ[key] = val
            print(f'✓ {key} loaded')
    except Exception:
        print(f'⚠ {key} not in Colab Secrets (optional)')
if lines:
    with open('.env', 'a') as f:
        f.write('\n'.join(lines) + '\n')
if 'WANDB_API_KEY' not in os.environ:
    os.environ['WANDB_DISABLED'] = 'true'
    print('wandb disabled (no key — training will still run)')

## 5. Mount Drive (for checkpoint backup)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/logic-zero/checkpoints/sft

## 6. Verify data files present

In [ ]:
import json
from collections import Counter
for f in ['data/sft_data.jsonl', 'data/dev_data.jsonl', 'data/eval_data.jsonl', 'data/eval_hashes.json']:
    assert os.path.exists(f), f'MISSING: {f}'
recs = [json.loads(l) for l in open('data/sft_data.jsonl')]
print(f'SFT records: {len(recs)}')
print(f'Per-n: {dict(sorted(Counter(r["n"] for r in recs).items()))}')
dev = [json.loads(l) for l in open('data/dev_data.jsonl')]
print(f'Dev records: {len(dev)}  (used for best-checkpoint selection)')

## 7. Resume from Drive (skip if first run)

If you ran SFT before and saved a checkpoint to Drive, restore it here so this notebook can pick up after Colab disconnect.

In [ ]:
drive_ckpt = '/content/drive/MyDrive/logic-zero/checkpoints/sft/best'
local_ckpt = '/content/logic-zero/results/checkpoints/sft/best'
if os.path.isdir(drive_ckpt):
    os.makedirs(os.path.dirname(local_ckpt), exist_ok=True)
    !cp -r {drive_ckpt} {local_ckpt}
    print(f'✓ Restored checkpoint from Drive: {local_ckpt}')
    !ls -la {local_ckpt}
else:
    print('No prior checkpoint in Drive — will train from scratch.')

## 8. Run SFT training

Default config: 3 epochs, batch 4 × grad-accum 4 (effective batch 16), LoRA r=16 on Q/K/V/O, lr=2e-4.

Auto-picks **bf16** on L4/A100, **fp16** on T4. Saves the **best dev-accuracy** checkpoint to `results/checkpoints/sft/best/` (callback in `train/sft.py`).

**If OOM on T4**: rerun with `--batch-size 2 --grad-accum 8` (same effective batch, less peak VRAM).

In [ ]:
!mkdir -p logs
!python -u -m train.sft 2>&1 | tee logs/sft_train.log

## 9. Verify checkpoint + backup to Drive

In [ ]:
best = 'results/checkpoints/sft/best'
assert os.path.isdir(best), 'No checkpoint saved — training probably failed; check logs/sft_train.log'
!ls -la {best}
import json
meta = json.load(open(f'{best}/dev_acc.json'))
print(f'\nBest dev_acc: {meta["acc"]:.3f} at epoch {meta["epoch"]}')

!rm -rf /content/drive/MyDrive/logic-zero/checkpoints/sft/best
!cp -r results/checkpoints/sft/best /content/drive/MyDrive/logic-zero/checkpoints/sft/best
!cp logs/sft_train.log /content/drive/MyDrive/logic-zero/checkpoints/sft/sft_train.log
print('\n✓ Checkpoint backed up to Drive')

## 10. Smoke test: generate one response from the trained model

Sanity check that the LoRA adapter loads + produces formatted output.

In [ ]:
import torch
from peft import PeftModel
from train.common import load_base_model, to_chat, extract_answer, check_format

model, tok = load_base_model()
model = PeftModel.from_pretrained(model, 'results/checkpoints/sft/best')
model.eval()

dev = [json.loads(l) for l in open('data/dev_data.jsonl')]
rec = dev[5]
print('PUZZLE:\n' + rec['puzzle'][:400])
print('\nGROUND TRUTH:', rec['ground_truth'])

prompt = to_chat(tok, rec['puzzle'])
inputs = tok(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=400, do_sample=False,
                        pad_token_id=tok.eos_token_id)
resp = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print('\nMODEL RESPONSE:\n' + resp)
print(f'\ncheck_format: {check_format(resp)}')
print(f'extracted: {extract_answer(resp, n=len(rec["ground_truth"]))}')

## ✅ Task 7 done.

Outputs:
- `results/checkpoints/sft/best/`  (≈30MB LoRA adapter, local)
- `/content/drive/MyDrive/logic-zero/checkpoints/sft/best/`  (backup)
- Dev accuracy in `best/dev_acc.json`

### Next: Task 8 — evaluate on the 1530-puzzle eval set

```
!python -m eval.run_eval --adapter results/checkpoints/sft/best --out results/eval_sft.json
!python -m eval.run_eval --out results/eval_base.json  # base model, no adapter
```

### Sanity criteria (per plan §Task 10)
- If dev_acc < 25%: STOP — something is broken. Debug before DPO.
- If 25% ≤ dev_acc < 50%: proceed cautiously to DPO; expect DPO gain of +5-10%.
- If dev_acc ≥ 50%: pipeline looks healthy.